In [1]:
# from dotenv import load_dotenv
# load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join now. If you want a certificate, make sure you submit your project while submissions are still being accepted.


In [2]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=custom_instructions,
)

In [3]:
assistant.rag("How do I get a certificate?")

'You can get a certificate only if you finish the course with a **live cohort**.  \n\nCertificates are **not** awarded in the self-paced mode, because you need to **peer-review 3 capstones** after submitting your project, and that can only happen while the course is running.'

In [4]:
assistant.rag("Can I still join the course after it started?")

'Yes, you can still join the course after it has started. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [5]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from https://ollama.com/download for your OS:\n   - macOS: download the `.pkg`\n   - Windows: download the `.msi`\n   - Linux: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Start it locally by running:\n   ```bash\n   ollama run llama3\n   ```\n\n   This will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\n3. If you want to test the local server, run:\n   ```bash\n   curl http://localhost:11434\n   ```\n\nIf you’re using it from Python, install the client with:\n\n```bash\npip install ollama\n```\n\nand use:\n\n```python\nimport ollama\n\nresponse = ollama.chat(\n    model=\'llama3\',\n    messages=[{"role": "user", "content": your_prompt}]\n)\n\nprint(response[\'message\'][\'content\'])\n```'

In [6]:
assistant.rag("How do I run Olama locally?")

'I’m not sure what “Olama” refers to in the course FAQ context.\n\nThe FAQ does say you can run the course locally instead of Codespaces if you’re comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module. If you do run locally, document your setup and keep the environment reproducible.\n\nIf you meant a specific tool, tell me its exact name and I’ll check the context.'

In [7]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes — in many cases you can still join, but it depends on the course rules and whether enrollment is still open.\n\nA good next step is to check:\n- the course start/end dates\n- whether late enrollment is allowed\n- any prerequisites or waiting list\n- the registration deadline\n\nIf you want, I can help you draft a quick message to the course organizer asking if you can still join.'

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [9]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? discovered the course join late enrollment"}', call_id='call_Q9gfm1bawHmzKRf5KdeeBPKk', name='search', type='function_call', id='fc_0a0baa744d4803d1006a56bea4b83c819d84ee6b9da0f0a1df', namespace=None, status='completed')]

In [10]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [11]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [12]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [13]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf your goal is to get a certificate, the important part is to submit your project while submissions are still open. If you just want to learn, you can start anytime since the materials are available.'

In [14]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(839, 53)

In [15]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(usage.input_tokens, usage.output_tokens)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.00015765


In [16]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [17]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [23]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment late join course FAQ"}
function_call: search {"query":"course enrollment late join discovered course can I still join FAQ"}


In [24]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

One important note: if you want a certificate, you need to submit your project while submissions are still being accepted. Homework is not mandatory, but the capstone/project is what matters for the certificate.

If you want, I can also help you figure out how to start the course or what to do first.


In [25]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [26]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run local setup install Ollama"}
iteration #2...
function_call: search {"query":"Ollama serve localhost:11434 python client ollama chat model llama3"}
iteration #3...
ASSISTANT:
To run **Ollama locally**, do this:

1. **Install Ollama**
   - macOS: download the `.pkg` from https://ollama.com/download
   - Windows: download the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model and open a local chat session.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should see a response from Ollama.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   print(response['messa

'To run **Ollama locally**, do this:\n\n1. **Install Ollama**\n   - macOS: download the `.pkg` from https://ollama.com/download\n   - Windows: download the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and open a local chat session.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should see a response from Ollama.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error in a notebook, restart the server with:\n```bash\nnohup ollama serve > nohup.out 2>&1 &\n```\n\nIf you want, I can also show yo

In [27]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enrollment discovered course can I still join"}
function_call: search {"query":"course FAQ enrollment late join after course started"}
function_call: search {"query":"register course after start late admission FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you’ll need to submit your project while submissions are still being accepted.

You can also start whenever you want using the available videos and GitHub materials, but certificates are only available for the live cohort, not self-paced study.

Would you like help with how to get started or the course workflow?


'Yes — you can still join the course.\n\nIf you want a certificate, though, you’ll need to submit your project while submissions are still being accepted.\n\nYou can also start whenever you want using the available videos and GitHub materials, but certificates are only available for the live cohort, not self-paced study.\n\nWould you like help with how to get started or the course workflow?'

In [28]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment late join FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course if you just discovered it.

One important note: if you want to receive a certificate, you need to submit your project while submissions are still being accepted.

If you want, I can also help you figure out the best way to start the course or explain the certificate requirements in more detail. Is there anything else you’d like to explore?


'Yes — you can still join the course if you just discovered it.\n\nOne important note: if you want to receive a certificate, you need to submit your project while submissions are still being accepted.\n\nIf you want, I can also help you figure out the best way to start the course or explain the certificate requirements in more detail. Is there anything else you’d like to explore?'

In [29]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit definition chess queen's gambit opening"}
function_call: search {"query":"queen gambit FAQ chess opening"}
function_call: search {"query":"what is queen gambit"}
iteration #2...
ASSISTANT:
If you mean the chess opening, the **Queen’s Gambit** is a classic opening that starts with:

1. d4 d5  
2. c4

White offers the c-pawn to try to pull Black’s d-pawn away from the center and gain space/control. It’s called a “gambit” because White is technically offering a pawn, though Black often doesn’t keep it.

There are two main versions:
- **Queen’s Gambit Accepted**: Black takes the pawn with `dxc4`
- **Queen’s Gambit Declined**: Black does not take it and instead supports the center

If you meant something else by “queen gambit,” tell me and I’ll clarify. Want me to explain the ideas behind it or show a few common lines?


'If you mean the chess opening, the **Queen’s Gambit** is a classic opening that starts with:\n\n1. d4 d5  \n2. c4\n\nWhite offers the c-pawn to try to pull Black’s d-pawn away from the center and gain space/control. It’s called a “gambit” because White is technically offering a pawn, though Black often doesn’t keep it.\n\nThere are two main versions:\n- **Queen’s Gambit Accepted**: Black takes the pawn with `dxc4`\n- **Queen’s Gambit Declined**: Black does not take it and instead supports the center\n\nIf you meant something else by “queen gambit,” tell me and I’ll clarify. Want me to explain the ideas behind it or show a few common lines?'

In [30]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"Queen's Gambit chess opening"}
iteration #3...
function_call: search {"query":"queen's gambit chess"}
iteration #4...
ASSISTANT:
I couldn’t find a relevant course/FAQ entry for “queen gambit,” so I can’t answer that from the course materials.

If you meant a course topic or logistics question, feel free to rephrase it, and I can look it up. Are there other areas you want to explore?


'I couldn’t find a relevant course/FAQ entry for “queen gambit,” so I can’t answer that from the course materials.\n\nIf you meant a course topic or logistics question, feel free to rephrase it, and I can look it up. Are there other areas you want to explore?'